# 🏠 Singapore HDB Resale Price Analysis & Prediction
## Notebook 1 — Data Collection & Cleaning

**Author:** Ren &nbsp;|&nbsp; **Stack:** Python, pandas, scikit-learn, LightGBM, Optuna, Streamlit

### Project goal
Over 80% of Singapore's resident population lives in HDB (Housing & Development Board) public housing,
and resale flats are the largest housing market in the country. This project:

1. **Analyses** ~230k resale transactions (2017–present) with pandas,
2. **Builds & tunes** regression models to predict resale prices (target metric: RMSE),
3. Powers an **interactive Streamlit dashboard** where a user can estimate a flat's market value
   and hunt for **overpriced / underpriced** listings.

### Notebook series
| # | Notebook | Purpose |
|---|----------|---------|
| 1 | **Data collection & cleaning** *(this one)* | Acquire the data, audit quality, fix types |
| 2 | Exploratory data analysis | Understand what drives prices |
| 3 | Feature engineering | Build model-ready features incl. geospatial ones |
| 4 | Model building & tuning | Baselines → LightGBM → Optuna tuning |
| 5 | Over/under-priced analysis | Use the model as a valuation engine |

### Data source
Official resale transactions from **[data.gov.sg](https://data.gov.sg)**
(dataset `d_8b84c4ee58e3cfc0ece0d773c8ca6abc`, *Resale flat prices based on registration date from Jan-2017 onwards*),
fetched via the public Datastore API by [`sg_hdb_price_analysis/data/fetch.py`](../sg_hdb_price_analysis/data/fetch.py)
and cached at `data/raw/hdb_resale_prices_all.csv`. Re-running `make fetch` refreshes the cache.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Notebook lives in notebooks/ — project root is one level up.
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
FIGURES = ROOT / "reports" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (11, 5)
pd.set_option("display.max_columns", 40)

## 1. Load the raw data

The fetch script pages through the Datastore API (10k rows per call) and concatenates everything.
Here we load the cached CSV — the raw file is treated as **immutable**; every transformation
happens downstream in code, so the whole pipeline is reproducible.

In [2]:
raw = pd.read_csv(ROOT / "data" / "raw" / "hdb_resale_prices_all.csv", parse_dates=["month"])
print(f"Rows: {len(raw):,}   Columns: {raw.shape[1]}")
print(f"Coverage: {raw['month'].min():%Y-%m} → {raw['month'].max():%Y-%m}")
raw.head()

Rows: 232,614   Columns: 11
Coverage: 2017-01 → 2026-06


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price
0,2017-01-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44.0,Improved,1979,61 years 04 months,232000.0
1,2017-01-01,SEMBAWANG,4 ROOM,477,SEMBAWANG DR,01 TO 03,86.0,Model A2,2000,82 years 06 months,305000.0
2,2017-01-01,QUEENSTOWN,5 ROOM,52,STRATHMORE AVE,19 TO 21,110.0,Improved,2006,88 years 11 months,860000.0
3,2017-01-01,QUEENSTOWN,5 ROOM,18,DOVER CRES,19 TO 21,111.0,Improved,2003,85 years 08 months,800000.0
4,2017-01-01,QUEENSTOWN,5 ROOM,21,HOLLAND DR,04 TO 06,119.0,Standard,1975,57 years,790000.0


### Data dictionary

| Column | Meaning |
|--------|---------|
| `month` | Transaction registration month |
| `town` | HDB town (26 towns, e.g. TAMPINES, BISHAN) |
| `flat_type` | 1 ROOM … 5 ROOM, EXECUTIVE, MULTI-GENERATION |
| `block`, `street_name` | Address of the block |
| `storey_range` | Floor band, e.g. "10 TO 12" |
| `floor_area_sqm` | Internal floor area |
| `flat_model` | Construction model (Improved, Model A, DBSS, …) |
| `lease_commence_date` | Year the 99-year lease started |
| `remaining_lease` | Remaining lease as "NN years MM months" string |
| `resale_price` | **Target** — transacted price in SGD |

In [3]:
raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 232614 entries, 0 to 232613
Data columns (total 11 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   month                232614 non-null  datetime64[ns]
 1   town                 232614 non-null  object        
 2   flat_type            232614 non-null  object        
 3   block                232614 non-null  object        
 4   street_name          232614 non-null  object        
 5   storey_range         232614 non-null  object        
 6   floor_area_sqm       232614 non-null  float64       
 7   flat_model           232614 non-null  object        
 8   lease_commence_date  232614 non-null  int64         
 9   remaining_lease      232614 non-null  object        
 10  resale_price         232614 non-null  float64       
dtypes: datetime64[ns](1), float64(2), int64(1), object(7)
memory usage: 19.5+ MB


## 2. Data quality audit

Before any analysis we check the three classic issues: **missing values**, **duplicates** and **dtype problems**.

In [4]:
missing = raw.isna().sum()
print("Missing values per column:")
print(missing[missing > 0] if missing.any() else "  → none 🎉")

dup_full = raw.duplicated().sum()
print(f"\nFully duplicated rows: {dup_full:,} ({dup_full / len(raw):.2%})")

Missing values per column:
  → none 🎉

Fully duplicated rows: 314 (0.13%)


> **Note on duplicates** — identical rows are *legitimate* here: two identical units in the same block,
> same floor band and same month sell for the same rounded price surprisingly often. The dataset has no
> transaction ID, so we keep them — dropping would under-weight high-volume estates.

In [5]:
# Enforce numeric dtypes (the API returns everything as strings)
df = raw.copy()
for col in ["floor_area_sqm", "resale_price", "lease_commence_date"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("Rows with non-numeric values coerced to NaN:", df[["floor_area_sqm", "resale_price", "lease_commence_date"]].isna().sum().sum())
df.describe().round(1)

Rows with non-numeric values coerced to NaN: 0


,month,floor_area_sqm,lease_commence_date,resale_price
count,232614,232614.0,232614.0,232614.0
mean,2021-11-19 17:07:52.367097856,96.7,1996.5,530555.3
min,2017-01-01 00:00:00,31.0,1966.0,140000.0
25%,2019-09-01 00:00:00,81.0,1985.0,390000.0
50%,2021-12-01 00:00:00,93.0,1997.0,500000.0
75%,2024-03-01 00:00:00,112.0,2012.0,638000.0
max,2026-06-01 00:00:00,366.7,2022.0,1728000.0
std,NaN,24.0,14.4,189921.7


## 3. Sanity checks on ranges

Do the values make physical sense? We verify the extremes instead of blindly trusting `describe()`.

In [6]:
checks = {
    "price <= 0": (df["resale_price"] <= 0).sum(),
    "price > S$2m": (df["resale_price"] > 2_000_000).sum(),
    "area < 20 sqm": (df["floor_area_sqm"] < 20).sum(),
    "area > 300 sqm": (df["floor_area_sqm"] > 300).sum(),
    "lease before 1960": (df["lease_commence_date"] < 1960).sum(),
    "lease in the future": (df["lease_commence_date"] > df["month"].dt.year).sum(),
}
pd.Series(checks, name="violations").to_frame()

,violations
price <= 0,0
price > S$2m,0
area < 20 sqm,0
area > 300 sqm,1
lease before 1960,0
lease in the future,0


In [7]:
print("Most expensive transactions ever recorded:")
df.nlargest(5, "resale_price")[["month", "town", "flat_type", "floor_area_sqm", "storey_range", "resale_price"]]

Most expensive transactions ever recorded:


,month,town,flat_type,floor_area_sqm,storey_range,resale_price
229282,2026-04-01,BUKIT MERAH,5 ROOM,113.0,46 TO 48,1728000.0
224958,2026-02-01,QUEENSTOWN,5 ROOM,122.0,19 TO 21,1700000.0
208756,2025-06-01,QUEENSTOWN,5 ROOM,122.0,22 TO 24,1658888.0
232431,2026-06-01,QUEENSTOWN,5 ROOM,122.0,04 TO 06,1650000.0
226254,2026-03-01,BUKIT MERAH,5 ROOM,112.0,25 TO 27,1648888.0


The S$1.5m+ records are real — million-dollar flats (high-floor, central, large units) are a
well-documented phenomenon in recent years, so these are **signal, not noise**. No row removal is needed:
the dataset arrives remarkably clean, which is typical for government-curated open data.

## 4. Categorical consistency

In [8]:
print(f"Towns ({df['town'].nunique()}):  {sorted(df['town'].unique())[:6]} …")
print(f"\nFlat types ({df['flat_type'].nunique()}):")
print(df["flat_type"].value_counts().to_string())
print(f"\nFlat models: {df['flat_model'].nunique()} unique  |  Storey ranges: {df['storey_range'].nunique()} unique")

Towns (26):  ['ANG MO KIO', 'BEDOK', 'BISHAN', 'BUKIT BATOK', 'BUKIT MERAH', 'BUKIT PANJANG'] …

Flat types (7):
flat_type
4 ROOM              98741
5 ROOM              56902
3 ROOM              55407
EXECUTIVE           16571
2 ROOM               4821
1 ROOM                 86
MULTI-GENERATION       86

Flat models: 21 unique  |  Storey ranges: 17 unique


Categories are already uppercase-normalised with no spelling variants — no cleaning required.

## 5. Summary

✅ **232k+ transactions, 2017 → present, no missing values, no invalid rows.**
The only transformation needed was numeric dtype coercion. Cleaning logic lives in the package
(`build_features` re-applies dtype fixes), so notebooks and the dashboard share one code path.

**Next →** [Notebook 2: Exploratory Data Analysis](02_exploratory_data_analysis.ipynb)